# Prepare KITTI input cache once — CPU session
Latest revision: **A2-LITE-INPUT-CACHE-2026-10-14-r1**

This notebook prepares reusable input data, NOT a new model experiment. Run it once on a CPU Colab runtime, after the current experiment has finished. Do not switch the active training session to CPU: its local data may be lost when the runtime is replaced.

It uses the exact reviewed images, labels, calibration and Chen train3712/val3769 identities. All7,481 frames are cached so future training and validation can share one input archive without mixing their split lists. No models, teacher predictions, checkpoints or previous experiment outputs are included.

The one-time job copies/verifies the22,443 files with four I/O workers, packs locally using ZIP_STORED (no PNG recompression), and writes ONE large archive plus a small identity receipt to Drive. Every input hash is checked against the original512 pilot manifest. Archive upload can resume verified64-MiB chunks after interruption. Ordinary dependencies and GPU software are not installed or changed.

Budget local space for both the unpacked data and one dataset-sized archive. The first preparation still takes Drive I/O time; the savings occur in later sessions, which copy one large file and extract locally. No measured speedup is claimed yet.

If the current experiment already has all data staged locally and has finished, you may instead opt into ALLOW_GPU_PREP=True in that same session and point DATASET at that verified folder. The preparation uses CPU/disk only; this avoids repeating the small-file Drive reads but still occupies that session during packaging. Default is CPU-only.

## 1. Drive, paths and all helpers
PILOT must be the ORIGINAL512-image recovery folder/ZIP. CACHE is a dedicated reusable input folder, not an experiment output. Keep it for subsequent experiments.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, shlex, subprocess, sys
REVISION='A2-LITE-INPUT-CACHE-2026-10-14-r1'
MOBILE_REPO=Path('/content/mobile_adas3d')
PILOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_recovery/a2_lite_640_recovery_kd_r1')
CACHE=Path('/content/drive/MyDrive/mobile_adas3d_inputs/kitti_chen_a2_v1')
PYTHON=Path(sys.executable);SCRIPT=MOBILE_REPO/'scripts/cache_a2_lite_inputs.py'
LOGS=CACHE/'operation_logs';LOGS.mkdir(parents=True,exist_ok=True)
def run_logged(command,name,cwd=None):
    command=list(map(str,command));log_path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',log_path,flush=True);tail=deque(maxlen=40)
    with log_path.open('w') as log:
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        finally:process.stdout.close()
        code=process.wait();log.write(f'\nEXIT CODE: {code}\n');log.flush()
    if code:raise RuntimeError(f'Exit {code}; log={log_path}\n'+'\n'.join(tail))
    return log_path
print(REVISION,'\nReusable input cache:',CACHE)
ALLOW_GPU_PREP=False
DATASET=Path('/content/kitti_a2_lite_accuracy_cache_prepare_v1')
LOCAL_ARCHIVE=Path('/content/a2_lite_archive_build/kitti_chen_7481.zip')
CANDIDATES=[Path(p) for p in (
    '/content/kitti_a2_lite_accuracy_geometry_a2_lite_512_geometry_kd_r1',
    '/content/kitti_a2_lite_accuracy_full_a2_lite_640_full_recovery_kd_r1',
    '/content/kitti_a2_lite_accuracy_recovery_a2_lite_640_recovery_kd_r1',
    '/content/kitti','/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti')]
READY=False


## 2. Update main and run cache-only tests
Use Runtime → Change runtime type → CPU for this separate preparation notebook. No packages need installing. If a GPU is detected, the default stops before data work; opt in only deliberately to reuse already-local data after training has finished.


In [ ]:
READY=False
try:
    gpu=subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True)
    allocated_gpu=bool(gpu.returncode==0 and gpu.stdout.strip())
except FileNotFoundError:allocated_gpu=False
if allocated_gpu and not ALLOW_GPU_PREP:
    raise RuntimeError('A GPU is allocated. Use a separate CPU runtime, or deliberately set ALLOW_GPU_PREP=True to reuse already-local data after training finishes.')
if allocated_gpu:print('Explicit GPU-session reuse enabled; cache preparation itself uses CPU/disk only.')
if not MOBILE_REPO.exists():run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use a clean main checkout.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only'],'update',MOBILE_REPO)
if not SCRIPT.is_file():raise FileNotFoundError('Pull the published cache workflow from main')
if not PILOT.exists():raise FileNotFoundError('Set PILOT to the original512 folder/ZIP in section1')
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_input_cache.py','-v'],'cache_tests',MOBILE_REPO)
READY=True


## 3. Prepare, verify and publish the reusable archive
Resumable input staging and archive transfer. Existing completed caches are read-only and must pass their identity/hash checks. A conflicting file stops rather than overwriting data. Current experiment manifests and checkpoints are never edited.


In [ ]:
if not READY:raise RuntimeError('Run sections1 and2 first')
command=[PYTHON,'-u',SCRIPT,'prepare','--pilot',PILOT,'--cache',CACHE,'--dataset',DATASET,'--local-archive',LOCAL_ARCHIVE,'--workers','4']
for candidate in CANDIDATES:command += ['--candidate',candidate]
run_logged(command,'prepare_input_archive',MOBILE_REPO)
RECEIPT=CACHE/'kitti_input_cache.json'
receipt=json.loads(RECEIPT.read_text())
assert receipt['complete'] and receipt['info']['train_frames']==3712 and receipt['info']['val_frames']==3769
print(json.dumps({'cache_id':receipt['cache_id'],'archive_size_GiB':receipt['archive']['size_bytes']/1024**3,'archive_sha256':receipt['archive']['sha256'],'train_frames':receipt['info']['train_frames'],'val_frames':receipt['info']['val_frames']},indent=2))


## 4. Return the small receipt, not the large dataset archive
Keep kitti_chen_7481.zip on Drive. Return kitti_input_cache.json for review. In the next experiment, run KITTI_A2_Input_Cache_Restore_Colab.ipynb first in its GPU session, with DATASET set to that experiment's new local dataset path. Then use that local data in the experiment's normal preparation stage. No current model experiment has changed.


In [ ]:
RECEIPT=CACHE/'kitti_input_cache.json'
if not RECEIPT.is_file():raise RuntimeError('Preparation has not completed; preserve operation_logs and rerun the failed section')
print('Archive ready:',CACHE/'kitti_chen_7481.zip')
print('Next notebook: notebooks/KITTI_A2_Input_Cache_Restore_Colab.ipynb')
from google.colab import files
files.download(str(RECEIPT))
